# Retrained DepthG-DepthPro-monocular vs CUPS-official baseline — Side-by-side

Four-column comparison per frame:

1. **RGB** — Cityscapes train left image, half-resolution.
2. **CUPS-official semantic** — from `cups_pseudo_labels_official_fullseq/` (DepthG + ZoeDepth + CRF, only strasbourg+zurich landed).
3. **CUPS-official instance** — from same root (~100 % empty; documented in `reports/2026-06-03_1208_depthg_depthpro_retrain.md`).
4. **OUR retrained DepthG semantic** — from `cups_pseudo_labels_depthg_depthpro_monocular/train/<city>/`. Generated by `mbps_pytorch/gen_semantic_only_depthg_depthpro.py` with the `epoch6_step1680.ckpt` retrained ckpt.

**What this tells us:** whether the cluster_probe mIoU = 14.8 gap to CUPS (22.3) shows up as *qualitatively worse* labels or just as a Hungarian-alignment shortfall. The reading is:
* Big block-y mis-segmentation → worse pseudo-labels.
* Coherent regions, different cluster IDs vs CUPS → same information, different partition (acceptable for Stage-2 training, which retrains a classification head from scratch).

**Note:** generation is long-running. Cells degrade gracefully when our cache is partial — they render a "pending" placeholder rather than failing.

In [ ]:
from __future__ import annotations

import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

CUPS_BASELINE_ROOT = Path("/Volumes/code_files_2/cityscapes_sequences/cups_official_root/cups_pseudo_labels_official_fullseq")
RETRAIN_SEM_ROOT = Path("/Volumes/code_files/datasets/cityscapes/cups_pseudo_labels_depthg_depthpro_monocular/train")
RGB_ROOT = Path("/Volumes/code_files/datasets/cityscapes/leftImg8bit/train")
RGB_FALLBACK_SEQ = Path("/Volumes/code_files_2/cityscapes_sequences/cups_official_root/Cityscapes/leftImg8bit_sequence/train")

random.seed(0)
np.random.seed(0)

for d in (CUPS_BASELINE_ROOT, RGB_ROOT):
    assert d.is_dir(), f"missing root: {d}"
if not RETRAIN_SEM_ROOT.is_dir():
    print(f"⚠ retrained-semantic root not yet populated: {RETRAIN_SEM_ROOT}")
    print("   the column 4 cells will draw a 'pending' placeholder until the gen script finishes.")


## 1. Inventory

The CUPS-official cache only covers strasbourg + zurich. Our retrain covers all 18 train cities.

In [ ]:
def list_baseline_stems() -> list[str]:
    sem = {p.name[: -len("_semantic.png")] for p in CUPS_BASELINE_ROOT.glob("*_semantic.png")}
    inst = {p.name[: -len("_instance.png")] for p in CUPS_BASELINE_ROOT.glob("*_instance.png")}
    return sorted(sem & inst)

def list_retrain_stems_by_city() -> dict[str, list[str]]:
    out: dict[str, list[str]] = {}
    if not RETRAIN_SEM_ROOT.is_dir():
        return out
    for city in sorted(p.name for p in RETRAIN_SEM_ROOT.iterdir() if p.is_dir()):
        out[city] = sorted(p.name[: -len("_semantic.png")] for p in (RETRAIN_SEM_ROOT / city).glob("*_semantic.png"))
    return out

baseline_stems = list_baseline_stems()
retrain_by_city = list_retrain_stems_by_city()
print(f"CUPS-official       : {len(baseline_stems)} frames (strasbourg + zurich)")
print(f"Our retrain         : {sum(len(v) for v in retrain_by_city.values())} frames across {len(retrain_by_city)} cities")
for c, fs in retrain_by_city.items():
    print(f"  {c:>20s}: {len(fs):4d}")


## 2. Palettes (shared across all panels for stable colour assignment)

In [ ]:
def make_palette(n: int = 27, seed: int = 7) -> np.ndarray:
    rng = np.random.default_rng(seed)
    return rng.integers(20, 240, size=(n, 3), dtype=np.uint8)

SEM_PALETTE = make_palette(27)

def colorize_semantic(idx: np.ndarray) -> np.ndarray:
    idx = np.clip(idx.astype(np.int32), 0, SEM_PALETTE.shape[0] - 1)
    return SEM_PALETTE[idx]

def colorize_instance(idx: np.ndarray, seed: int = 11) -> np.ndarray:
    ids = np.unique(idx)
    rng = np.random.default_rng(seed)
    cmap = {0: np.array([0, 0, 0], dtype=np.uint8)}
    for i in ids:
        if i == 0:
            continue
        cmap[int(i)] = rng.integers(40, 240, size=3, dtype=np.uint8)
    out = np.zeros((*idx.shape, 3), dtype=np.uint8)
    for i, color in cmap.items():
        out[idx == i] = color
    return out

def overlay(rgb: np.ndarray, mask_rgb: np.ndarray, alpha: float = 0.55) -> np.ndarray:
    a, b = rgb.astype(np.float32), mask_rgb.astype(np.float32)
    return np.clip((1 - alpha) * a + alpha * b, 0, 255).astype(np.uint8)

def placeholder(h: int, w: int, text: str) -> np.ndarray:
    img = np.full((h, w, 3), 220, dtype=np.uint8)
    return img


## 3. Per-frame loader (4-column quartet)

Looks up RGB → CUPS-official semantic → CUPS-official instance → our semantic.
Falls back to a placeholder for our column when the gen hasn't reached that city yet.

In [ ]:
def find_rgb(stem: str) -> Path | None:
    city = stem.split("_")[0]
    candidates = [
        RGB_ROOT / city / f"{stem}.png",
        RGB_FALLBACK_SEQ / city / f"{stem}.png",
    ]
    for c in candidates:
        if c.exists():
            return c
    return None

def find_our_sem(stem: str) -> Path | None:
    city = stem.split("_")[0]
    p = RETRAIN_SEM_ROOT / city / f"{stem}_semantic.png"
    return p if p.exists() else None

def load_quartet(stem: str, alpha: float = 0.55):
    rgb_path = find_rgb(stem)
    if rgb_path is None:
        return None
    cups_sem_path = CUPS_BASELINE_ROOT / f"{stem}_semantic.png"
    cups_inst_path = CUPS_BASELINE_ROOT / f"{stem}_instance.png"
    our_sem_path = find_our_sem(stem)

    cups_sem = np.array(Image.open(cups_sem_path)) if cups_sem_path.exists() else None
    cups_inst = np.array(Image.open(cups_inst_path)) if cups_inst_path.exists() else None
    our_sem = np.array(Image.open(our_sem_path)) if our_sem_path is not None else None

    target_hw = cups_sem.shape if cups_sem is not None else our_sem.shape if our_sem is not None else (640, 1280)
    rgb = np.array(Image.open(rgb_path).convert("RGB").resize((target_hw[1], target_hw[0]), Image.BILINEAR))

    cups_sem_overlay = overlay(rgb, colorize_semantic(cups_sem), alpha) if cups_sem is not None else placeholder(*target_hw, "missing CUPS semantic")
    cups_inst_overlay = overlay(rgb, colorize_instance(cups_inst), alpha) if cups_inst is not None else placeholder(*target_hw, "missing CUPS instance")
    our_sem_overlay = overlay(rgb, colorize_semantic(our_sem), alpha) if our_sem is not None else placeholder(*target_hw, "pending")
    return rgb, cups_sem_overlay, cups_inst_overlay, our_sem_overlay, our_sem is not None, cups_inst is not None


## 4. Single-frame quartet

In [ ]:
def show_quartet(stem: str, alpha: float = 0.55) -> None:
    q = load_quartet(stem, alpha)
    if q is None:
        print(f"no RGB for {stem}")
        return
    rgb, cs, ci, os_overlay, has_our, has_inst = q
    fig, ax = plt.subplots(1, 4, figsize=(22, 4))
    ax[0].imshow(rgb); ax[0].set_title(f"RGB — {stem}")
    ax[1].imshow(cs); ax[1].set_title("CUPS-official semantic")
    ax[2].imshow(ci); ax[2].set_title(f"CUPS-official instance ({'has' if has_inst else 'missing'})")
    ax[3].imshow(os_overlay); ax[3].set_title(f"Ours: retrained DepthG semantic ({'ready' if has_our else 'pending'})")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()

if baseline_stems:
    show_quartet(baseline_stems[0])
    show_quartet(baseline_stems[len(baseline_stems) // 2])
    show_quartet(baseline_stems[-1])


## 5. Random grid: strasbourg + zurich (where direct comparison is possible)

In [ ]:
def show_grid(stems: list[str], n: int = 5, alpha: float = 0.55) -> None:
    picks = random.sample(stems, k=min(n, len(stems)))
    fig, axes = plt.subplots(n, 4, figsize=(22, 4 * n))
    if n == 1:
        axes = axes[None, :]
    for row, stem in enumerate(picks):
        q = load_quartet(stem, alpha)
        if q is None:
            continue
        rgb, cs, ci, ours, has_our, has_inst = q
        axes[row, 0].imshow(rgb); axes[row, 0].set_title(stem, fontsize=8)
        axes[row, 1].imshow(cs); axes[row, 1].set_title("CUPS semantic", fontsize=8)
        axes[row, 2].imshow(ci); axes[row, 2].set_title("CUPS instance", fontsize=8)
        axes[row, 3].imshow(ours); axes[row, 3].set_title("Our semantic", fontsize=8)
        for a in axes[row]: a.axis("off")
    plt.tight_layout(); plt.show()

if baseline_stems:
    show_grid(baseline_stems, n=5)


## 6. Cities where we have new pseudo-labels but CUPS-official has none

In [ ]:
if retrain_by_city:
    new_only = [c for c in retrain_by_city if c not in ("strasbourg", "zurich") and retrain_by_city[c]]
    print(f"cities with new pseudo-labels not in CUPS-official: {new_only}")
    if new_only:
        city = random.choice(new_only)
        sample = retrain_by_city[city][0]
        show_quartet(sample)


## 7. Reading the comparison

- **CUPS semantic vs Ours semantic, same frame**: the absolute cluster IDs will differ (no shared label space — both are unsupervised). What matters is whether the *partitioning* is coherent (large connected regions on stuff, distinct regions on things) and whether the boundaries follow the underlying image.
- **If our semantic looks fragmented or random**: the cluster mIoU = 14.8 reflects genuine label noise, and we should re-retrain (option B from the report: `depth_loss_decay=false`) before downstream training.
- **If our semantic looks coherent but assigns different IDs to road/building/sky**: the gap is just Hungarian alignment, and Stage-2 Cascade Mask R-CNN will absorb this fine (it relearns its classification head from scratch on the pseudo-labels).

Provenance: ckpt `checkpoints/depthg_depthpro_monocular/epoch6_step1680.ckpt`, log `logs/gen_semantic_only_*.log`, report `reports/2026-06-03_1208_depthg_depthpro_retrain.md`.